# FirstWave — ML Pipeline Walkthrough

This notebook walks through every step of the FirstWave model:
1. Load synthetic prescriber data
2. Engineer features (including KOL PageRank)
3. Train XGBoost adoption classifier
4. Train Cox PH survival model for timing
5. Generate SHAP explanations
6. Evaluate with lift curves and ROC

Run `python data/generate_synthetic_data.py` first to create the data.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import xgboost as xgb
import shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from lifelines import KaplanMeierFitter, CoxPHFitter
import matplotlib.pyplot as plt

df = pd.read_parquet('../data/processed/physician_features.parquet')
print(f'Loaded {len(df):,} physicians')
df.head()

## 1. Class balance

In [ ]:
print(f"Early adopter rate: {df['early_adopter'].mean():.1%}")
df.groupby('specialty')['early_adopter'].mean().sort_values(ascending=False)

## 2. Train/test split

In [ ]:
FEATURE_COLS = [
    'years_in_practice', 'rural', 'prior_injectable_glp1_prescriber',
    'prior_sglt2_prescriber', 'prior_dpp4_prescriber', 'patient_panel_size',
    'unique_drugs_prescribed', 'total_open_payments_usd', 'num_speaker_events',
    'is_speaker', 'kol_pagerank_normalized', 'has_kol_connection',
]
X = pd.concat([df[FEATURE_COLS], pd.get_dummies(df[['specialty', 'state', 'gender']])], axis=1)
y = df['early_adopter']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f'Train: {len(X_train):,}  Test: {len(X_test):,}')

## 3. Train XGBoost

In [ ]:
pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

model = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    scale_pos_weight=pos_weight, random_state=42, eval_metric='auc'
)
model.fit(X_train, y_train)
y_pred = model.predict_proba(X_test)[:, 1]
print(f'ROC-AUC: {roc_auc_score(y_test, y_pred):.4f}')

## 4. SHAP explanations

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test.iloc[:100])
shap.summary_plot(shap_values, X_test.iloc[:100], plot_type='bar')

## 5. Lift curve

In [ ]:
lift_df = pd.DataFrame({'true': y_test.values, 'score': y_pred})
lift_df = lift_df.sort_values('score', ascending=False).reset_index(drop=True)
lift_df['decile'] = pd.qcut(lift_df.index, 10, labels=False) + 1
baseline = lift_df['true'].mean()
lift = lift_df.groupby('decile')['true'].mean() / baseline

plt.figure(figsize=(10, 5))
plt.bar(range(1, 11), lift.values, color='#f97306')
plt.axhline(y=1, color='red', linestyle='--', label='Random baseline')
plt.xlabel('Decile (1 = highest predicted)')
plt.ylabel('Lift over baseline')
plt.title(f'Top decile lift: {lift.iloc[0]:.2f}x')
plt.legend()
plt.show()

## 6. Survival analysis

In [ ]:
kmf = KaplanMeierFitter()
fig, ax = plt.subplots(figsize=(10, 6))
for specialty in ['Endocrinology', 'Internal Medicine', 'Family Medicine']:
    mask = df['specialty'] == specialty
    kmf.fit(df.loc[mask, 'months_to_event'], df.loc[mask, 'event_observed'], label=specialty)
    kmf.plot_survival_function(ax=ax)
plt.title('Time to first Rybelsus prescription by specialty')
plt.xlabel('Months from launch')
plt.ylabel('Probability of not yet prescribed')
plt.show()